# Recall@k when scores tie

A bike-share company's help center answers riders' questions from twelve short notes,
retrieved with TF-IDF as in the lesson. Before trusting the retriever, the team measures
recall@k on ten test questions, each with one note that holds its answer. TF-IDF gives a
score of exactly 0 to every note that shares no word with a question, so many scores are
equal, and the order of equal scores is whatever the sort happens to produce. You will
compute a rank that no ordering of the ties can improve, and the recall@k that a random
ordering of the ties gives on average.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import re
import numpy as np

notes = [
    "Unlock a bike by scanning the code on its handlebar with the app.",                    # 0
    "A ride costs one dollar to unlock plus fifteen cents per minute.",                     # 1
    "Annual members ride free for the first 45 minutes of every trip.",                     # 2
    "Return a bike by pushing it into an empty dock until the light turns green.",          # 3
    "When every dock at a station is full, the app shows the nearest station with space.",  # 4
    "Electric bikes have a battery range of about 40 miles.",                               # 5
    "Report a broken bike with the wrench button on the dock.",                             # 6
    "Riders must be at least 16 years old.",                                                # 7
    "Helmets are not provided, but we recommend wearing one.",                              # 8
    "Stations stay open all year, though some close during heavy snow.",                    # 9
    "A bike that is not returned within 24 hours costs a fee of 1200 dollars.",             # 10
    "A group pass covers up to five riders on one account.",                                # 11
]
questions = [                       # (question, the note that holds its answer)
    ("How much does a ride cost?", 1),
    ("What if the dock is full?", 4),
    ("How far can an electric bike go?", 5),
    ("Can my kid come along?", 7),
    ("Do you lend helmets?", 8),
    ("What happens if I keep the bike?", 10),
    ("Who fixes flat tires?", 6),
    ("Should I bring my own helmet on the ride?", 8),
    ("Are stations open in winter?", 9),
    ("My bike is broken", 6),
]

tokenize = lambda s: re.findall(r"[a-z0-9]+", s.lower())
vocab = sorted({w for d in notes for w in tokenize(d)})
col = {w: j for j, w in enumerate(vocab)}

def counts(text):
    v = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in col:
            v[col[w]] += 1
    return v

tf = np.array([counts(d) for d in notes])
idf = np.log(len(notes) / (tf > 0).sum(axis=0)) + 1
D = tf * idf
D /= np.linalg.norm(D, axis=1, keepdims=True)
Q = np.array([counts(q) for q, _ in questions]) * idf
lengths = np.linalg.norm(Q, axis=1, keepdims=True)
Q = np.divide(Q, lengths, out=np.zeros_like(Q), where=lengths > 0)   # no known word: the row stays 0

S = Q @ D.T                                   # S[i, j] is the cosine of question i with note j
answers = np.array([a for _, a in questions])
print(f"{S.shape[0]} questions, {S.shape[1]} notes, {np.sum(S == 0)} of the {S.size} scores are exactly 0")

## Part 1 (coding task): The worst rank of the answer note

Write `worst_rank(S, answers)`. `S` is an $m \times n$ array of scores, one row
per question and one column per note, and `answers[i]` is the column of question
`i`'s answer note. A retriever sorts each row from the highest score to the
lowest, and it may put notes with equal scores in any order. Return an integer
array whose entry `i` is the worst position that question `i`'s answer note can
end up in, counting the first position as 1. A whole-array approach is the neat
one, and a loop over the questions also passes the check but is slower.

In [ ]:
def worst_rank(S, answers):
    """For each row, the worst position of its answer note over every order of equal scores."""
    S = np.asarray(S, dtype=float)
    answers = np.asarray(answers)
    # TODO
    raise NotImplementedError

In [ ]:
S1 = np.array([[0.9, 0.5, 0.5, 0.1],
               [0.9, 0.5, 0.5, 0.1],
               [0.0, 0.0, 0.0, 0.0],
               [0.2, 0.7, 0.4, 0.7]])
r = np.asarray(worst_rank(S1, [0, 1, 2, 3]))
assert r.shape == (4,), f"expected one rank per row, shape (4,), got {r.shape}"
assert np.issubdtype(r.dtype, np.integer), "return whole-number positions"
assert r[0] == 1, f"the highest note of row 0 got position {r[0]}: count the first position as 1"
assert r[2] == 4, f"row 2 is all zeros, so the answer note ties with every note and its worst position is 4, not {r[2]}"
assert r[1] == 3, f"in row 1 the answer note ties with note 2, which may be put ahead of it: its worst position is 3, not {r[1]}"
assert np.array_equal(r, [1, 3, 4, 2]), f"got {r.tolist()}, expected [1, 3, 4, 2]"
i = np.arange(600).reshape(40, 15)
S_big = ((i * 37) % 11) / 10                   # 11 distinct scores, so every row has ties
a_big = (np.arange(40) * 7) % 15
assert np.asarray(worst_rank(S_big, a_big)).sum() == 345, "wrong positions on a 40 x 15 array with many ties"
print("worst positions on the help center:", worst_rank(S, answers))
print("Part 1 passed")

## Part 2 (coding task): Expected recall@k under a random order of ties

Write `expected_recall(S, answers, k)`. Now suppose the retriever puts notes with equal
scores in a uniformly random order, chosen afresh for each question. Return the expected
recall@k: the expected fraction of the questions whose answer note is among the first `k`
notes of its row.

In [ ]:
def expected_recall(S, answers, k):
    """Expected share of rows whose answer note is in the first k when ties are ordered at random."""
    S = np.asarray(S, dtype=float)
    answers = np.asarray(answers)
    # TODO
    raise NotImplementedError

In [ ]:
S1 = np.array([[0.9, 0.5, 0.5, 0.1],
               [0.9, 0.5, 0.5, 0.1],
               [0.0, 0.0, 0.0, 0.0],
               [0.2, 0.7, 0.4, 0.7]])
assert np.isclose(expected_recall([[0.9, 0.5, 0.1]], [0], 2), 1.0), "the answer note is first for sure, so it is in the top 2 with probability 1: a probability cannot exceed 1"
assert np.isclose(expected_recall([[0.5, 0.5, 0.5]], [1], 1), 1 / 3), "three notes tie for first place, so the answer note is first one time in three"
e2 = expected_recall(S1, [0, 1, 2, 3], 2)
assert not np.isclose(e2, 0.5), "this counts a row only when its worst position is in the top k: a random order puts a tied answer note there part of the time"
assert not np.isclose(e2, 1.0), "this counts every tied answer note as retrieved: a random order puts it in the top k only part of the time"
assert np.isclose(e2, 0.75), f"expected recall@2 of 0.75 on the four rows above, got {e2}"
assert np.isclose(expected_recall(S1, [0, 1, 2, 3], 1), 0.4375), "recall@1 on the four rows above should be (1 + 0 + 1/4 + 1/2) / 4"
assert np.isclose(expected_recall([[0.9, 0.8, 0.3, 0.3]], [3], 1), 0.0), "two notes always come before the answer note, so it is never first: a probability cannot be negative"
assert np.isclose(expected_recall(S1, [0, 1, 2, 3], 10), 1.0), "with k above the number of notes every answer note is in the top k: a probability cannot exceed 1"
i = np.arange(600).reshape(40, 15)
S_big = ((i * 37) % 11) / 10
a_big = (np.arange(40) * 7) % 15
assert np.isclose(expected_recall(S_big, a_big, 5), 0.3375), "wrong expected recall@5 on a 40 x 15 array with many ties"

own = S[np.arange(len(S)), answers][:, None]
best = (S > own).sum(axis=1) + 1               # the position if every tie went the answer note's way
worst = worst_rank(S, answers)
print("k   ties in its favor   ties against it   random order of ties")
for k in (1, 2, 3):
    print(f"{k}   {np.mean(best <= k):17.3f}   {np.mean(worst <= k):15.3f}   {expected_recall(S, answers, k):20.3f}")
print("Part 2 passed")

## Solution

### Part 1: The worst rank of the answer note

In [ ]:
def worst_rank(S, answers):
    """For each row, the worst position of its answer note over every order of equal scores."""
    S = np.asarray(S, dtype=float)
    answers = np.asarray(answers)
    own = S[np.arange(len(S)), answers]          # the answer note's score in each row
    return (S >= own[:, None]).sum(axis=1)       # notes scoring at least as high, itself included

**Why it works.** The worst case puts every note that ties with the answer note ahead of it, so its
position is the number of notes scoring strictly higher, plus the tied ones, plus itself.
That is the number of notes scoring at least as high as the answer note, the answer note
included, which `S >= own[:, None]` counts for every row at once. Taking the position from
`np.argsort` instead would break the ties in one fixed order. With a stable sort
(`np.argsort(..., kind="stable")`) that order is the note order, and an answer note
that sits early in the list would get a better position than the worst one.

### Part 2: Expected recall@k under a random order of ties

In [ ]:
def expected_recall(S, answers, k):
    """Expected share of rows whose answer note is in the first k when ties are ordered at random."""
    S = np.asarray(S, dtype=float)
    answers = np.asarray(answers)
    own = S[np.arange(len(S)), answers][:, None]
    above = (S > own).sum(axis=1)                # notes certain to come first
    level = (S == own).sum(axis=1)               # notes tied with the answer note, itself included
    # The answer note is equally likely to take each of the positions above + 1, ..., above + level.
    return float(np.mean(np.clip((k - above) / level, 0, 1)))

**Why it works.** Every note scoring strictly higher than the answer note comes before it in every order,
so with `above` such notes and `level` notes tied with it (itself included), its position
is equally likely to be any of `above + 1` to `above + level`. The chance that it is at
most `k` is `(k - above) / level`, clipped to 0 when the notes above already fill the top
`k` and to 1 when the whole tied block fits. The mean of these chances over the questions
is the expected recall@k, because the expectation of a mean is the mean of the expectations.

On the help center, recall@1 is 0.70 if every tie goes the answer note's way,
0.50 if every tie goes against it, and 0.517 with a random order. The gap comes
from two questions. "Can my kid come along?" and "Who fixes flat tires?" share
no word with any note, so every score in their rows is 0. Counting ties in the
answer's favor makes both perfect hits, although the retriever knows nothing
about them, and a random order gives each a chance of 1/12 at k = 1. "Should I
bring my own helmet on the ride?" also scores its answer note 0, because
"helmet" is not "helmets", but seven notes score above it, so at k = 1 it is a
miss in all three columns of the table above, and no tie ordering can save it.
The lesson's `np.argsort(-scores)` breaks the ties in one fixed order rather than the
worst case. Which order that is depends on the sort: with a stable sort
(`np.argsort(-scores, kind="stable")`) it is the note order, so the recall it reports
for such rows depends on where the answer note happens to sit in the list. Report
the worst case or the expected value, and say which.